# 📏 RAGAS: Evaluating RAG Systems with Ollama

**RAGAS** (Retrieval Augmented Generation Assessment) gives your RAG pipeline a report card — four scores that together tell you *where it's failing*, without needing human judges.

| Metric | Question it answers | Inputs needed |
|--------|--------------------|-|
| **Faithfulness** | Is the answer supported by the retrieved context? | answer + contexts |
| **Answer Relevancy** | Does the answer actually address the question? | question + answer |
| **Context Precision** | Are the retrieved chunks useful (not noise)? | question + contexts + ground truth |
| **Context Recall** | Did we retrieve *all* the relevant information? | contexts + ground truth |

**What we'll build in this notebook:**
1. A Space Facts RAG system (FAISS + Ollama `llama3.2`)
2. An evaluation dataset with known ground truths
3. Full RAGAS evaluation → a scored results table

**Model**: `llama3.2` (3B — fast, runs on CPU)  
**RAG embeddings**: `all-MiniLM-L6-v2` (local, ~80 MB)  
**RAGAS evaluation LLM**: Ollama `llama3.2`

> 🔁 Run cells top-to-bottom. The RAG pipeline runs first, then RAGAS scores it.


## 📦 Install dependencies

In [ ]:
!pip install -q ragas==0.1.21 langchain-ollama langchain datasets \
         sentence-transformers faiss-cpu ollama
print("✅ Dependencies ready")

## 🦙 Ollama setup

If you're running **locally**, start Ollama in a terminal:

```bash
ollama serve            # in one terminal
ollama pull llama3.2    # pull the model once (~2 GB)
ollama pull nomic-embed-text   # embeddings for RAGAS answer-relevancy
```

If you're on **Google Colab**, the cell below installs + starts Ollama automatically.


In [ ]:
import subprocess, time, os, shutil

def is_colab():
    try:
        import google.colab; return True
    except ImportError: return False

if is_colab():
    print("Colab detected — installing Ollama...")

    subprocess.run("apt-get install -y zstd", shell=True, check=True,
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    subprocess.run("curl -fsSL https://ollama.ai/install.sh | sh",
                   shell=True, check=True,
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    # Start Ollama server in background
    subprocess.Popen(["ollama", "serve"],
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(3)
    print("  ✅ Ollama server started")

    # Pull models
    for model in ["llama3.2", "nomic-embed-text"]:
        print(f"  ⬇️  Pulling {model}…")
        subprocess.run(f"ollama pull {model}", shell=True, check=True,
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        print(f"  ✅ {model} ready")
else:
    print("Local environment detected.")
    print("Make sure you've run: ollama pull llama3.2 && ollama pull nomic-embed-text")

print("\n🦙 Ollama ready!")

## 🔧 Imports & configuration

In [ ]:
import json, warnings
import numpy as np
import pandas as pd

# RAG pipeline
from sentence_transformers import SentenceTransformer
import faiss
import ollama as ollama_client

# RAGAS
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_ollama import ChatOllama, OllamaEmbeddings

warnings.filterwarnings("ignore")

# ── Model config ──────────────────────────────────────────────────────────────
OLLAMA_MODEL   = "llama3.2"      # generation + RAGAS evaluation
EMBED_MODEL    = "all-MiniLM-L6-v2"  # RAG retrieval embeddings (local)
OLLAMA_EMBED   = "nomic-embed-text"  # RAGAS answer-relevancy embeddings
OLLAMA_URL     = "http://localhost:11434"

print(f"LLM          : {OLLAMA_MODEL}")
print(f"RAG embeds   : {EMBED_MODEL}")
print(f"RAGAS embeds : {OLLAMA_EMBED}")
print("✅ Config done")

## 📚 Knowledge base — Space Facts

A small factual corpus. RAGAS evaluation works best when we have *ground truth* answers we can verify against the corpus — space facts are perfect for this.


In [ ]:
CORPUS = [
    "Mercury is the smallest planet in the Solar System and the closest to the Sun. "
    "It has no moons and no significant atmosphere. A day on Mercury (one rotation) "
    "lasts about 59 Earth days, but a year (orbit around the Sun) only takes 88 Earth days.",

    "Mars has two small moons named Phobos and Deimos. The planet has the tallest volcano "
    "in the Solar System, Olympus Mons, which stands about 22 km high. Mars has a thin "
    "atmosphere composed mainly of carbon dioxide.",

    "The James Webb Space Telescope (JWST) was launched on December 25, 2021. It observes "
    "the universe in infrared light and can see farther back in time than any previous "
    "telescope. JWST is located at the L2 Lagrange point, about 1.5 million km from Earth.",

    "Saturn is famous for its ring system, which consists mainly of ice particles and rocky "
    "debris. Saturn has 146 confirmed moons — the most of any planet in the Solar System. "
    "Its largest moon, Titan, has a thick nitrogen atmosphere and liquid methane lakes.",

    "The Apollo 11 mission landed the first humans on the Moon on July 20, 1969. Astronauts "
    "Neil Armstrong and Buzz Aldrin walked on the lunar surface while Michael Collins orbited "
    "above in the command module. They returned safely to Earth on July 24, 1969.",

    "The speed of light in a vacuum is approximately 299,792 km/s. Light from the Sun takes "
    "about 8 minutes and 20 seconds to reach Earth. The nearest star to the Sun, Proxima "
    "Centauri, is about 4.24 light-years away.",
]

print(f"📄 Corpus: {len(CORPUS)} documents")
for i, doc in enumerate(CORPUS):
    print(f"  [{i}] {doc[:70]}…")

## 🛠️ RAG pipeline

Build a standard embed → index → retrieve → generate pipeline. We'll run this to produce *answers* and *contexts* for the evaluation dataset.


In [ ]:
# ── Build FAISS index ────────────────────────────────────────────────────────
print("Building embedding index…")
embedder = SentenceTransformer(EMBED_MODEL)
doc_embeddings = embedder.encode(CORPUS, show_progress_bar=False)

index = faiss.IndexFlatIP(doc_embeddings.shape[1])
norms = np.linalg.norm(doc_embeddings, axis=1, keepdims=True)
index.add(doc_embeddings / norms)  # normalise for cosine similarity

print(f"✅ Index built — {index.ntotal} vectors")

def retrieve(query: str, top_k: int = 2) -> list[str]:
    """Return top-k most relevant documents for a query."""
    q_emb = embedder.encode([query])
    q_emb = q_emb / np.linalg.norm(q_emb)
    _, ids = index.search(q_emb, top_k)
    return [CORPUS[i] for i in ids[0]]

def generate(question: str, contexts: list[str]) -> str:
    """Generate an answer given a question and retrieved context."""
    context_text = "\n\n".join(f"[Doc {i+1}] {c}" for i, c in enumerate(contexts))
    prompt = (
        f"Answer the question using ONLY the information in the context below.\n"
        f"If the context doesn't contain the answer, say 'I don't know.'\n\n"
        f"Context:\n{context_text}\n\n"
        f"Question: {question}\n\n"
        f"Answer:"
    )
    response = ollama_client.chat(
        model=OLLAMA_MODEL,
        messages=[{"role": "user", "content": prompt}],
    )
    return response["message"]["content"].strip()

# Quick smoke test
test_ctx = retrieve("What is the tallest volcano in the Solar System?")
test_ans = generate("What is the tallest volcano in the Solar System?", test_ctx)
print(f"\n🧪 Smoke test answer:\n  {test_ans[:120]}")

## 🧪 Create the evaluation dataset

RAGAS needs four things for each example:
- `question` — what we asked
- `answer` — what the RAG system said
- `contexts` — what was retrieved (list of strings)
- `ground_truth` — the correct answer (for precision/recall scoring)

We define `questions` + `ground_truths` by hand, then *run our RAG pipeline* to fill in `answers` and `contexts` automatically.


In [ ]:
# ── Define questions with known ground truths ─────────────────────────────
QA_PAIRS = [
    {
        "question": "How many moons does Saturn have?",
        "ground_truth": "Saturn has 146 confirmed moons, the most of any planet.",
    },
    {
        "question": "When was the James Webb Space Telescope launched?",
        "ground_truth": "The James Webb Space Telescope was launched on December 25, 2021.",
    },
    {
        "question": "Who were the astronauts on Apollo 11 who walked on the Moon?",
        "ground_truth": "Neil Armstrong and Buzz Aldrin walked on the Moon during Apollo 11.",
    },
    {
        "question": "How long does light from the Sun take to reach Earth?",
        "ground_truth": "Light from the Sun takes about 8 minutes and 20 seconds to reach Earth.",
    },
    {
        "question": "What is the atmosphere of Mars primarily made of?",
        "ground_truth": "Mars has a thin atmosphere composed mainly of carbon dioxide.",
    },
]

# ── Run RAG to generate answers + retrieve contexts ───────────────────────
print("Running RAG pipeline for each question…")
questions, answers, contexts, ground_truths = [], [], [], []

for qa in QA_PAIRS:
    ctx  = retrieve(qa["question"], top_k=2)
    ans  = generate(qa["question"], ctx)

    questions.append(qa["question"])
    answers.append(ans)
    contexts.append(ctx)
    ground_truths.append(qa["ground_truth"])

    print(f"  Q: {qa['question'][:60]}")
    print(f"  A: {ans[:80]}\n")

print("✅ Evaluation dataset ready")

## ⚙️ Configure RAGAS with Ollama

RAGAS wraps any LangChain-compatible LLM. We pass our local Ollama models through `LangchainLLMWrapper` and `LangchainEmbeddingsWrapper`.


In [ ]:
# ── Wire up Ollama → RAGAS ────────────────────────────────────────────────
ragas_llm = LangchainLLMWrapper(
    ChatOllama(model=OLLAMA_MODEL, base_url=OLLAMA_URL)
)
ragas_embeddings = LangchainEmbeddingsWrapper(
    OllamaEmbeddings(model=OLLAMA_EMBED, base_url=OLLAMA_URL)
)

print(f"✅ RAGAS LLM        → Ollama {OLLAMA_MODEL}")
print(f"✅ RAGAS Embeddings → Ollama {OLLAMA_EMBED}")

## 📊 Run RAGAS evaluation

This calls the LLM several times per example (once per metric that needs reasoning). On a CPU with `llama3.2` expect ~2–5 minutes for 5 questions.


In [ ]:
# ── Build HuggingFace Dataset ─────────────────────────────────────────────
eval_dataset = Dataset.from_dict({
    "question":     questions,
    "answer":       answers,
    "contexts":     contexts,
    "ground_truth": ground_truths,
})

print(f"Dataset: {len(eval_dataset)} rows")
print(f"Columns: {eval_dataset.column_names}\n")

# ── Run evaluation ────────────────────────────────────────────────────────
print("⏳ Running RAGAS evaluation (this may take a few minutes)…")

result = evaluate(
    eval_dataset,
    metrics=[
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall,
    ],
    llm=ragas_llm,
    embeddings=ragas_embeddings,
)

print("\n✅ Evaluation complete!")
print(result)

## 📈 Results — per-question breakdown

In [ ]:
# ── Overall scores ───────────────────────────────────────────────────────
scores_df = result.to_pandas()

print("=" * 65)
print(f"{'Metric':<22} {'Score':>8}   {'Interpretation'}")
print("=" * 65)

metric_info = {
    "faithfulness":      ("1.0 = fully grounded in context",   "⚠️ < 0.7 = hallucination risk"),
    "answer_relevancy":  ("1.0 = perfectly on-topic",          "⚠️ < 0.7 = drifts from question"),
    "context_precision": ("1.0 = retrieved docs all relevant",  "⚠️ < 0.7 = too much noise"),
    "context_recall":    ("1.0 = retrieved everything needed",  "⚠️ < 0.7 = missing key info"),
}

for metric, (good, bad) in metric_info.items():
    if metric in scores_df.columns:
        avg = scores_df[metric].mean()
        flag = "✅" if avg >= 0.7 else "❌"
        hint = good if avg >= 0.7 else bad
        print(f"  {flag} {metric:<20} {avg:>6.3f}   {hint}")

print("=" * 65)
print()

# ── Per-question table ────────────────────────────────────────────────────
display_cols = ["question"] + [c for c in scores_df.columns if c in metric_info]
short_q = scores_df["question"].str[:45] + "…"
display_df = scores_df[display_cols].copy()
display_df["question"] = short_q
print(display_df.to_string(index=False, float_format="{:.3f}".format))

---
## 🔬 Deep dive: what each metric actually does

### Faithfulness
RAGAS asks the LLM: *"Is each claim in this answer supported by the context?"*  
It breaks the answer into atomic claims, checks each one, and returns `supported / total`.

```
Answer: "Saturn has 146 moons and a thick atmosphere."
                        ↓
Claims: ["Saturn has 146 moons", "Saturn has a thick atmosphere"]
Check:  ✅ supported by context          ❌ not in context
Score:  1 / 2 = 0.5
```

**Low faithfulness** → your LLM is adding facts not in the retrieved docs (hallucination).

---

### Answer Relevancy
RAGAS generates *reverse questions* from your answer and measures how similar they are to the original question (cosine similarity of embeddings).

```
Answer: "It was launched on December 25, 2021."
Reverse questions generated:
  → "When was it launched?"   similarity: 0.92
  → "What happened in 2021?"  similarity: 0.71
Average similarity → relevancy score
```

**Low answer relevancy** → your answer is technically correct but doesn't address what was asked.

---

### Context Precision
Checks whether the *top-ranked* retrieved documents are the *most useful* ones.  
Computed using the ground truth: did the relevant chunks appear first, or buried at the bottom?

**Low context precision** → retrieval is noisy; you're fetching irrelevant docs alongside useful ones.

---

### Context Recall
Compares the ground truth against what was retrieved: are all the facts in the ground truth *present* in the retrieved context?

**Low context recall** → your retrieval is missing key documents; the LLM can't answer well because the evidence was never fetched.

---

### 🎯 Quick diagnostic guide

| Symptom | Likely culprit |
|---------|---------------|
| Bad answers with good context | ↓ Faithfulness → LLM ignoring context |
| Correct answer, wrong question | ↓ Answer Relevancy → prompt drift |
| Too many retrieved docs, poor quality | ↓ Context Precision → k too large, reranking needed |
| Important info missing from answer | ↓ Context Recall → retrieval strategy needs improvement |


## 🔍 Find the weakest examples

In [ ]:
import warnings; warnings.filterwarnings("ignore")

metrics_to_check = [c for c in scores_df.columns if c in metric_info]
if not metrics_to_check:
    print("No metric columns found — rerun the evaluation cell.")
else:
    for metric in metrics_to_check:
        worst_idx = scores_df[metric].idxmin()
        row = scores_df.iloc[worst_idx]
        print(f"{'─'*65}")
        print(f"  Lowest {metric}: {row[metric]:.3f}")
        print(f"  Q: {row['question']}")
        print(f"  A: {answers[worst_idx][:120]}")
        print(f"  GT: {ground_truths[worst_idx]}")
        print()

## 🧨 Experiment: intentionally degrade retrieval

What happens to RAGAS scores when we retrieve *wrong* documents on purpose? This shows how RAGAS catches retrieval failures.


In [ ]:
import random

print("Running RAG with random (wrong) context…")
bad_answers, bad_contexts = [], []

for qa in QA_PAIRS:
    # Pick 2 random documents that are NOT the best match
    shuffled = random.sample(CORPUS, 2)
    ans  = generate(qa["question"], shuffled)
    bad_answers.append(ans)
    bad_contexts.append(shuffled)

bad_dataset = Dataset.from_dict({
    "question":     questions,
    "answer":       bad_answers,
    "contexts":     bad_contexts,
    "ground_truth": ground_truths,
})

print("⏳ Evaluating degraded RAG…")
bad_result = evaluate(
    bad_dataset,
    metrics=[faithfulness, context_precision, context_recall],
    llm=ragas_llm,
    embeddings=ragas_embeddings,
)

bad_df = bad_result.to_pandas()
good_means = scores_df[metrics_to_check].mean()
bad_means  = bad_df[[c for c in bad_df.columns if c in metric_info]].mean()

print("\n📊 Good retrieval vs random retrieval:")
print(f"{'Metric':<25} {'Good':>8}  {'Random':>8}  {'Delta':>8}")
print("-" * 55)
for m in bad_means.index:
    if m in good_means.index:
        delta = bad_means[m] - good_means[m]
        arrow = "⬇️" if delta < -0.05 else ("⬆️" if delta > 0.05 else "≈")
        print(f"  {m:<23} {good_means[m]:>8.3f}  {bad_means[m]:>8.3f}  {delta:>+8.3f} {arrow}")

---
## 🏁 Summary

| Metric | What a low score means | How to fix it |
|--------|----------------------|---------------|
| **Faithfulness** | LLM adds info not in context | Stricter prompts, smaller `k`, better context quality |
| **Answer Relevancy** | Answer drifts off-topic | Tune the generation prompt, adjust temperature |
| **Context Precision** | Retrieved docs contain noise | Reduce `k`, add re-ranking step |
| **Context Recall** | Missing relevant documents | Increase `k`, improve embeddings, try hybrid search |

### Next steps

- 🔬 Try **Advanced RAG Techniques** to improve precision/recall → `rag-techniques-ollama.ipynb`
- 📊 Add more questions to your evaluation dataset for statistically reliable scores
- 🔄 Build a CI loop: commit code → run RAG → run RAGAS → fail if score drops below threshold
- 📈 Track scores over time to detect regressions as you improve your pipeline

---
*Built with ❤️ by [Crunch](https://github.com/Copilotclaw/copilotclaw) — the Copilotclaw AI agent*
